# Universe D pair diagnosis

Full frozen-stack scorecard for Universe D share-class pairs (`WSO|WSO.B`, `HEI|HEI.A`, `NWS|NWSA`).
Uses `run_s2_backtest` + `US_ALPACA_D_REALISTIC` costs. Does **not** change STARs.

**Helpers:** `04_backtest/s2_coint/diagnosis.py` (`stack_scorecard`, `pair_deployment_table`, `plotly_pair_diagnosis`)

In [ ]:
from __future__ import annotations

import os
import sys

import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.diagnosis import (
    check_fill_timing,
    pair_deployment_table,
    plotly_pair_diagnosis,
    print_extreme_trades,
    stack_scorecard,
)
from backtest.s2_coint.research import (
    DEFAULT_STAR_STACK,
    frozen_pairs_for_universe,
    load_s1_weekly,
    load_star_stack,
    load_universe_panels,
    research_is_end_for,
)

USE_OOS = False
COMPARE_BASELINE_COSTS = True

stack = load_star_stack(DEFAULT_STAR_STACK)
universe = str(stack["UNIVERSE_STAR"])
bar = str(stack.get("BAR_STAR") or "1d")
pair_ids = frozen_pairs_for_universe(universe, bar=bar)
is_end = research_is_end_for(universe)
train, full = load_universe_panels(universe, bar, pair_ids)
panel = full if USE_OOS else train
s1_weekly = load_s1_weekly()

realistic = stack_scorecard(
    panel,
    stack,
    use_oos=USE_OOS,
    is_end=is_end,
    s1_weekly=s1_weekly,
    cost_profile="US_ALPACA_D_REALISTIC",
)
print("=== Realistic D costs ===")
print(realistic["metrics"])
display(pair_deployment_table(realistic["pair_trades"]))

if COMPARE_BASELINE_COSTS:
    baseline = stack_scorecard(
        panel,
        stack,
        use_oos=USE_OOS,
        is_end=is_end,
        s1_weekly=s1_weekly,
        cost_profile="US_ALPACA",
    )
    print("\n=== Baseline US_ALPACA (no borrow) ===")
    print(baseline["metrics"])

print_extreme_trades(realistic["pair_trades"], n=3)
timing = check_fill_timing(realistic["pair_trades"], panel)
display(timing)

for pid in pair_ids:
    g = panel.loc[panel["pair_id"] == pid]
    t = realistic["pair_trades"].loc[realistic["pair_trades"]["pair_id"] == pid]
    fig = plotly_pair_diagnosis(
        g,
        t,
        entry_z=float(realistic["config"].entry_z),
        z_window=int(realistic["config"].z_window),
        pair_returns=realistic["pair_returns"].get(pid),
        is_end=is_end,
        title=f"{pid} (full stack, realistic costs)",
    )
    fig.show()